# DataMind Survey: 3-Hour A100 Run

This notebook runs the narrowed, feasible scope for the DataMind survey project on one Colab Pro A100.

| # | Paper connection | Runnable experiment |
|---|---|---|
| Exp 1 | Zhu et al. Table 1, table-info ablation | QRData filenames only vs. filenames plus column names, dtypes, and 3 sample rows |
| Exp 2 | Zhu et al. Table 4 and Figure 2, code/error analysis | Categorize wrong baseline Exp-1 trajectories without extra model calls |
| Exp 3 | Lightweight inference analogue of Section 4.3/Table 5 turn-length results | Same model and samples, max ReAct turns in `{2, 4, 6}` |

This does not run fine-tuning, DiscoveryBench, a 14B model, or a paid API judge. Use `Runtime -> Change runtime type -> GPU -> A100` before running.


In [ ]:
# === Configuration ===
REPO_URL = "https://github.com/Razim021/datamind-survey.git"
BRANCH = "main"

# Run a smoke test first; flip to False for final numbers.
SMOKE_TEST = True

if SMOKE_TEST:
    N_EXP1 = 5
    N_EXP3 = 5
    EXP1_BUDGET_S = 600
    EXP3_BUDGET_S = 900
else:
    N_EXP1 = 40
    N_EXP3 = 25
    EXP1_BUDGET_S = 1800
    EXP3_BUDGET_S = 1800

MODEL_NAME = "Qwen2.5-7B-Instruct"
HF_MODEL = "Qwen/Qwen2.5-7B-Instruct"
API_PORT = 8000
MAX_ROUNDS_EXP1 = 4
TURN_BUDGETS = "2,4,6"

print(f"SMOKE_TEST={SMOKE_TEST}  N_EXP1={N_EXP1}  N_EXP3={N_EXP3}")


In [ ]:
# === GPU check ===
!nvidia-smi | head -20


In [ ]:
# === Clone repo, install deps, fetch QRData only ===
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/datamind-survey')
if not PROJECT_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH])

%cd /content/datamind-survey

if not Path('Datamind-main').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/zjunlp/DataMind', 'Datamind-main'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'vllm'])
subprocess.check_call([sys.executable, 'download_data.py', '--dataset', 'qrdata'])

os.environ['DATAMIND_ROOT'] = '/content/datamind-survey/Datamind-main'
os.environ['DATA_DIR'] = '/content/datamind-survey/data'
os.environ['JUDGE_BACKEND'] = 'local'
print('Setup complete')


In [ ]:
# === Sanity-check QRData loader ===
from experiments.utils.data_loader import load_qrdata
samples = load_qrdata('data')
print('QRData samples:', len(samples))
print('Example question:', samples[0]['question'][:160])
print('Example files:', samples[0]['file_paths'][:2])


In [ ]:
# === Start or reuse vLLM server ===
import subprocess
import time
import urllib.request
from pathlib import Path

log_path = Path('/content/vllm.log')

def vllm_ready():
    try:
        with urllib.request.urlopen(f'http://localhost:{API_PORT}/v1/models', timeout=2):
            return True
    except Exception:
        return False

if vllm_ready():
    print('vLLM is already ready; reusing the existing server.')
else:
    server_cmd = [
        'vllm', 'serve', HF_MODEL,
        '--served-model-name', MODEL_NAME,
        '--port', str(API_PORT),
        '--dtype', 'float16',
        '--max-model-len', '4096',
        '--gpu-memory-utilization', '0.85',
    ]
    server_log = open(log_path, 'w')
    vllm_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)
    print('Started vLLM, pid =', vllm_proc.pid, ' log:', log_path)

    for _ in range(120):
        if vllm_ready():
            print('vLLM is ready')
            break
        time.sleep(5)
    else:
        print('vLLM did not start. Last log lines:')
        for line in log_path.read_text(errors='replace').splitlines()[-40:]:
            print(line)
        raise RuntimeError('vLLM not ready')


## Experiment 1: Table-Metadata Ablation

Runs the Table 1 style ablation from Zhu et al.: the model gets either filenames only or filenames plus compact table metadata. Metrics are accuracy and code-error rate.


In [ ]:
import os
import subprocess
import sys

os.chdir('/content/datamind-survey/experiments')
subprocess.check_call([
    sys.executable, 'run_exp1_comprehension.py',
    '--model_name', MODEL_NAME,
    '--data_dir', '../data',
    '--dataset', 'qrdata',
    '--sub_experiment', 'info',
    '--n_samples', str(N_EXP1),
    '--api_port', str(API_PORT),
    '--max_rounds', str(MAX_ROUNDS_EXP1),
    '--judge_backend', 'local',
    '--time_budget_s', str(EXP1_BUDGET_S),
    '--output_dir', 'results/exp1_colab',
])


## Experiment 2: Error Categorization

Reads only the baseline Exp-1 result file (`qrdata_wo_info_*.json`) and categorizes wrong answers. This costs no extra GPU time.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, 'run_exp2_code_analysis.py',
    '--mode', 'categorize',
    '--results_dir', 'results/exp1_colab',
    '--file_glob', 'qrdata_wo_info_*.json',
    '--n_errors', str(N_EXP1),
    '--judge_backend', 'local',
    '--output_dir', 'results/exp2_colab',
])


## Experiment 3: Turn-Budget Sweep

Tests whether simply allowing more inference-time ReAct turns helps. This is a lightweight analogue of the paper's turn-length discussion, not a fine-tuned reproduction of Table 5.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, 'run_exp3_turn_budget.py',
    '--model_name', MODEL_NAME,
    '--data_dir', '../data',
    '--dataset', 'qrdata',
    '--turn_budgets', TURN_BUDGETS,
    '--n_samples', str(N_EXP3),
    '--api_port', str(API_PORT),
    '--judge_backend', 'local',
    '--time_budget_s', str(EXP3_BUDGET_S),
    '--output_dir', 'results/exp3_colab',
])


## Print Summaries


In [ ]:
import glob
import json
for result_path in sorted(glob.glob('results/*/summary.json') + glob.glob('results/*/error_categories.json')):
    print()
    print('===', result_path, '===')
    print(json.dumps(json.load(open(result_path)), indent=2))


## Optional Upgrade: 14B Exp 1 Only

Run these cells after the 7B full run if you have time. They stop the 7B vLLM server, start Qwen2.5-14B-Instruct, and rerun only the table-metadata ablation into `results/exp1_14b_colab`. This adds a direct 7B-vs-14B scaling comparison without fine-tuning or extra datasets.


In [ ]:
# === Stop existing vLLM server before loading 14B ===
import subprocess
import time
import urllib.request

try:
    vllm_proc.terminate()
    vllm_proc.wait(timeout=60)
    print('Stopped previous vLLM process:', vllm_proc.pid)
except NameError:
    print('No vllm_proc variable found; trying pkill fallback.')
except subprocess.TimeoutExpired:
    vllm_proc.kill()
    vllm_proc.wait(timeout=30)
    print('Killed previous vLLM process:', vllm_proc.pid)

# Fallback for Drive notebooks that lost the Python process handle.
subprocess.run(['pkill', '-f', 'vllm serve'], check=False)
time.sleep(10)

def port_open():
    try:
        with urllib.request.urlopen(f'http://localhost:{API_PORT}/v1/models', timeout=2):
            return True
    except Exception:
        return False

print('Port still serving after stop:', port_open())


In [ ]:
# === Start Qwen2.5-14B-Instruct vLLM server ===
import subprocess
import time
import urllib.request
from pathlib import Path

MODEL_NAME_14B = 'Qwen2.5-14B-Instruct'
HF_MODEL_14B = 'Qwen/Qwen2.5-14B-Instruct'
log_path_14b = Path('/content/vllm_14b.log')

def vllm_14b_ready():
    try:
        with urllib.request.urlopen(f'http://localhost:{API_PORT}/v1/models', timeout=2):
            return True
    except Exception:
        return False

server_cmd_14b = [
    'vllm', 'serve', HF_MODEL_14B,
    '--served-model-name', MODEL_NAME_14B,
    '--port', str(API_PORT),
    '--dtype', 'float16',
    '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.90',
]
server_log_14b = open(log_path_14b, 'w')
vllm_proc_14b = subprocess.Popen(server_cmd_14b, stdout=server_log_14b, stderr=subprocess.STDOUT)
print('Started 14B vLLM, pid =', vllm_proc_14b.pid, ' log:', log_path_14b)

for _ in range(180):  # up to 15 min for download/load
    if vllm_14b_ready():
        print('14B vLLM is ready')
        break
    time.sleep(5)
else:
    print('14B vLLM did not start. Last log lines:')
    for line in log_path_14b.read_text(errors='replace').splitlines()[-60:]:
        print(line)
    raise RuntimeError('14B vLLM not ready')


In [ ]:
# === Optional Exp 1 rerun with Qwen2.5-14B-Instruct ===
import os
import subprocess
import sys

os.chdir('/content/datamind-survey/experiments')
subprocess.check_call([
    sys.executable, 'run_exp1_comprehension.py',
    '--model_name', MODEL_NAME_14B,
    '--data_dir', '../data',
    '--dataset', 'qrdata',
    '--sub_experiment', 'info',
    '--n_samples', str(N_EXP1),
    '--api_port', str(API_PORT),
    '--max_rounds', str(MAX_ROUNDS_EXP1),
    '--judge_backend', 'local',
    '--time_budget_s', str(EXP1_BUDGET_S),
    '--output_dir', 'results/exp1_14b_colab',
])


In [ ]:
# === Print upgraded summaries and re-zip all results ===
import glob
import json
for result_path in sorted(glob.glob('results/*/summary.json') + glob.glob('results/*/error_categories.json')):
    print()
    print('===', result_path, '===')
    print(json.dumps(json.load(open(result_path)), indent=2))

subprocess.check_call(['zip', '-r', '/content/datamind_first3_results.zip', 'results'], stdout=subprocess.DEVNULL)
print('Updated /content/datamind_first3_results.zip with 14B Exp 1 results.')


In [ ]:
# === Zip results for download ===
%cd /content/datamind-survey/experiments
!zip -r /content/datamind_first3_results.zip results > /dev/null
!ls -lh /content/datamind_first3_results.zip
print('Download /content/datamind_first3_results.zip from the Colab file panel.')
